# 📈 RL Stock Trader — Deep Q-Network (DQN)

A self-contained RL agent that learns to trade a single stock using a **Double Dueling DQN**.

### Architecture
- **Dueling DQN** — separate value + advantage heads for stable Q-estimates
- **Double DQN** — decoupled action selection/evaluation to reduce overestimation
- **Experience Replay** — random minibatch sampling from a fixed-size buffer
- **ε-greedy exploration** with exponential decay

### Action Space (5 actions)
| Action | Description |
|--------|-------------|
| 0 | Hold |
| 1 | Buy 50% of available cash |
| 2 | Sell 50% of holdings |
| 3 | Buy All |
| 4 | Sell All |

Run cells **top to bottom**. On Colab, run the install cell first.

## 1 · Install Dependencies

In [ ]:
!pip install -q yfinance ta

## 2 · Imports

In [ ]:
import random, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import deque
from dataclasses import dataclass
from typing import List, Tuple

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

try:
    import yfinance as yf
    HAS_YF = True
except ImportError:
    HAS_YF = False
    print('yfinance not found — using synthetic data.')

try:
    import ta
    HAS_TA = True
except ImportError:
    HAS_TA = False
    print('ta not found — using basic indicators only.')


## 3 · Config

Edit any parameter here before running the rest of the notebook.

In [ ]:
@dataclass
class Config:
    # Data
    ticker:          str   = 'AAPL'
    start_date:      str   = '2018-01-01'
    end_date:        str   = '2023-12-31'
    train_ratio:     float = 0.8

    # Environment
    initial_cash:    float = 10_000.0
    transaction_fee: float = 0.001   # 0.1% per trade
    window_size:     int   = 20      # lookback steps

    # DQN
    hidden_size:     int   = 128
    lr:              float = 1e-3
    gamma:           float = 0.99
    epsilon_start:   float = 1.0
    epsilon_end:     float = 0.05
    epsilon_decay:   int   = 500
    batch_size:      int   = 64
    memory_size:     int   = 10_000
    target_update:   int   = 10

    # Training
    episodes:        int   = 200
    seed:            int   = 42


CFG = Config()

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


## 4 · Data & Technical Indicators

Downloads OHLCV data via `yfinance`. Falls back to synthetic GBM prices if the download fails.

In [ ]:
def _synthetic_ohlcv(cfg: Config, n: int = 1500) -> pd.DataFrame:
    """Geometric Brownian Motion prices with fake volume."""
    np.random.seed(cfg.seed)
    ret   = np.random.normal(0.0003, 0.015, n)
    price = 100.0 * np.cumprod(1 + ret)
    noise = lambda s: np.random.uniform(0, s, n)
    close = price
    open_ = close * (1 + noise(0.005) - 0.0025)
    high  = np.maximum(close, open_) * (1 + noise(0.008))
    low   = np.minimum(close, open_) * (1 - noise(0.008))
    vol   = np.random.randint(500_000, 5_000_000, n)
    dates = pd.date_range('2018-01-01', periods=n, freq='B')
    return pd.DataFrame({'open': open_, 'high': high, 'low': low,
                         'close': close, 'volume': vol}, index=dates)


def add_indicators(df: pd.DataFrame) -> pd.DataFrame:
    c  = df['close']
    df = df.copy()
    # Returns & volatility
    df['ret_1']  = c.pct_change(1)
    df['ret_5']  = c.pct_change(5)
    df['ret_20'] = c.pct_change(20)
    df['vol_20'] = df['ret_1'].rolling(20).std()
    # Moving averages (relative to price)
    df['sma_10'] = c.rolling(10).mean() / c - 1
    df['sma_30'] = c.rolling(30).mean() / c - 1
    df['sma_50'] = c.rolling(50).mean() / c - 1
    # RSI
    delta = c.diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    df['rsi'] = 100 - 100 / (1 + gain / (loss + 1e-9))
    # MACD
    ema12 = c.ewm(span=12, adjust=False).mean()
    ema26 = c.ewm(span=26, adjust=False).mean()
    macd  = ema12 - ema26
    df['macd']        = macd / c
    df['macd_signal'] = macd.ewm(span=9, adjust=False).mean() / c
    # Bollinger Bands
    mid = c.rolling(20).mean()
    std = c.rolling(20).std()
    df['bb_width'] = (2 * std) / (mid + 1e-9)
    df['bb_pos']   = (c - (mid - 2 * std)) / (4 * std + 1e-9)
    # Volume ratio
    df['vol_norm'] = df['volume'] / df['volume'].rolling(20).mean()
    if HAS_TA:
        import ta as _ta
        df['atr'] = _ta.volatility.AverageTrueRange(
            df['high'], df['low'], df['close'], window=14).average_true_range() / c
        df['cci'] = _ta.trend.CCIIndicator(
            df['high'], df['low'], df['close'], window=20).cci() / 200
    return df


def fetch_data(cfg: Config) -> pd.DataFrame:
    if HAS_YF:
        print(f'Downloading {cfg.ticker} {cfg.start_date} → {cfg.end_date}…')
        try:
            df = yf.download(cfg.ticker, start=cfg.start_date,
                             end=cfg.end_date, progress=False)
            df.columns = [c[0].lower() if isinstance(c, tuple) else c.lower()
                          for c in df.columns]
            df = df[['open', 'high', 'low', 'close', 'volume']].dropna()
        except Exception:
            df = pd.DataFrame()
        if df.empty:
            print('Download failed — using synthetic data.')
            df = _synthetic_ohlcv(cfg)
    else:
        df = _synthetic_ohlcv(cfg)
    df = add_indicators(df)
    df.dropna(inplace=True)
    print(f'Dataset: {len(df)} rows, {df.shape[1]} features')
    return df


def split_data(df, cfg):
    split = int(len(df) * cfg.train_ratio)
    return (df.iloc[:split].reset_index(drop=True),
            df.iloc[split:].reset_index(drop=True))


df = fetch_data(CFG)
train_df, test_df = split_data(df, CFG)
df.tail(3)


## 5 · Trading Environment

In [ ]:
class StockEnv:
    """
    Single-stock trading environment.
    Actions: 0=Hold  1=Buy50%  2=Sell50%  3=BuyAll  4=SellAll
    """
    N_ACTIONS = 5

    def __init__(self, df: pd.DataFrame, cfg: Config):
        self.df        = df
        self.cfg       = cfg
        self.feat_cols = [c for c in df.columns
                          if c not in ('open', 'high', 'low', 'close', 'volume')]
        self.obs_size  = len(self.feat_cols) * cfg.window_size + 2
        self.reset()

    def reset(self):
        self.t       = self.cfg.window_size
        self.cash    = self.cfg.initial_cash
        self.shares  = 0.0
        self.trades  = 0
        self.history = []
        return self._obs()

    def step(self, action):
        price = float(self.df['close'].iloc[self.t])
        before = self._portfolio(price)
        self._execute(action, price)
        self.t += 1
        done   = self.t >= len(self.df) - 1
        after  = self._portfolio(float(self.df['close'].iloc[self.t]))
        reward = (after - before) / (before + 1e-9)
        self.history.append(after)
        return self._obs(), reward, done

    def _execute(self, action, price):
        fee = self.cfg.transaction_fee
        if action == 1:
            spend        = self.cash * 0.5
            self.shares += spend / (price * (1 + fee))
            self.cash   -= spend;  self.trades += 1
        elif action == 2:
            sell         = self.shares * 0.5
            self.cash   += sell * price * (1 - fee)
            self.shares -= sell;   self.trades += 1
        elif action == 3:
            self.shares += self.cash / (price * (1 + fee))
            self.cash    = 0.0;    self.trades += 1
        elif action == 4:
            self.cash   += self.shares * price * (1 - fee)
            self.shares  = 0.0;    self.trades += 1

    def _portfolio(self, price):
        return self.cash + self.shares * price

    def _obs(self):
        w = self.df[self.feat_cols].iloc[
            self.t - self.cfg.window_size : self.t
        ].values.flatten().astype(np.float32)
        price = float(self.df['close'].iloc[self.t])
        total = self._portfolio(price) + 1e-9
        meta  = np.array([self.cash / total,
                           self.shares * price / total], dtype=np.float32)
        return np.nan_to_num(np.concatenate([w, meta]),
                             nan=0.0, posinf=1.0, neginf=-1.0)

    @property
    def final_value(self):
        return self._portfolio(float(self.df['close'].iloc[self.t]))


_env = StockEnv(train_df, CFG)
print(f'Observation size : {_env.obs_size}')
print(f'Action space     : {StockEnv.N_ACTIONS}')
print(f'Train rows       : {len(train_df)} | Test rows: {len(test_df)}')


## 6 · Dueling DQN Network

In [ ]:
class DQN(nn.Module):
    """Dueling DQN with LayerNorm."""
    def __init__(self, obs_size, n_actions, hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_size, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),   nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
        )
        self.value_head     = nn.Linear(hidden // 2, 1)
        self.advantage_head = nn.Linear(hidden // 2, n_actions)

    def forward(self, x):
        h = self.net(x)
        V = self.value_head(h)
        A = self.advantage_head(h)
        return V + A - A.mean(dim=-1, keepdim=True)


_net = DQN(_env.obs_size, StockEnv.N_ACTIONS, CFG.hidden_size)
print(_net)
print(f'\nTotal parameters: {sum(p.numel() for p in _net.parameters()):,}')


## 7 · Replay Buffer & Agent

In [ ]:
@dataclass
class Transition:
    state:      np.ndarray
    action:     int
    reward:     float
    next_state: np.ndarray
    done:       bool


class ReplayBuffer:
    def __init__(self, capacity):
        self.buf = deque(maxlen=capacity)
    def push(self, *args):
        self.buf.append(Transition(*args))
    def sample(self, n):
        return random.sample(self.buf, n)
    def __len__(self):
        return len(self.buf)


class DQNAgent:
    def __init__(self, obs_size, cfg):
        self.cfg    = cfg
        self.policy = DQN(obs_size, StockEnv.N_ACTIONS, cfg.hidden_size).to(DEVICE)
        self.target = DQN(obs_size, StockEnv.N_ACTIONS, cfg.hidden_size).to(DEVICE)
        self.target.load_state_dict(self.policy.state_dict())
        self.target.eval()
        self.optimizer = optim.Adam(self.policy.parameters(), lr=cfg.lr)
        self.memory    = ReplayBuffer(cfg.memory_size)
        self.steps = 0
        self.episode = 0

    @property
    def epsilon(self):
        return self.cfg.epsilon_end + (
            self.cfg.epsilon_start - self.cfg.epsilon_end
        ) * math.exp(-self.episode / self.cfg.epsilon_decay)

    def select_action(self, state):
        if random.random() < self.epsilon:
            return random.randrange(StockEnv.N_ACTIONS)
        with torch.no_grad():
            s = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
            return int(self.policy(s).argmax(dim=1).item())

    def push(self, *args):
        self.memory.push(*args);  self.steps += 1

    def learn(self):
        if len(self.memory) < self.cfg.batch_size:
            return None
        batch  = self.memory.sample(self.cfg.batch_size)
        S  = torch.FloatTensor(np.array([t.state      for t in batch])).to(DEVICE)
        A  = torch.LongTensor( np.array([t.action     for t in batch])).to(DEVICE)
        R  = torch.FloatTensor(np.array([t.reward     for t in batch])).to(DEVICE)
        S2 = torch.FloatTensor(np.array([t.next_state for t in batch])).to(DEVICE)
        D  = torch.FloatTensor(np.array([t.done       for t in batch])).to(DEVICE)
        with torch.no_grad():
            next_a  = self.policy(S2).argmax(dim=1, keepdim=True)
            next_q  = self.target(S2).gather(1, next_a).squeeze()
            target_q = R + self.cfg.gamma * next_q * (1 - D)
        current_q = self.policy(S).gather(1, A.unsqueeze(1)).squeeze()
        loss = F.smooth_l1_loss(current_q, target_q)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy.parameters(), 1.0)
        self.optimizer.step()
        return loss.item()

    def sync_target(self):
        self.target.load_state_dict(self.policy.state_dict())

    def save(self, path='best_model.pt'):
        torch.save(self.policy.state_dict(), path)
        print(f'Saved → {path}')

    def load(self, path='best_model.pt'):
        self.policy.load_state_dict(torch.load(path, map_location=DEVICE))
        self.sync_target()


print('ReplayBuffer and DQNAgent defined.')


## 8 · Training

Progress is printed every 20 episodes. The best checkpoint is saved to `best_model.pt`.

In [ ]:
train_env = StockEnv(train_df, CFG)
agent     = DQNAgent(train_env.obs_size, CFG)

ep_returns, ep_losses, ep_epsilons = [], [], []
best_return = -np.inf

print(f'Training {CFG.episodes} episodes on {DEVICE}…\n')

for ep in range(1, CFG.episodes + 1):
    state             = train_env.reset()
    total_loss, steps = 0.0, 0
    done              = False

    while not done:
        action                = agent.select_action(state)
        next_state, rew, done = train_env.step(action)
        agent.push(state, action, rew, next_state, done)
        state = next_state
        loss  = agent.learn()
        if loss is not None:
            total_loss += loss;  steps += 1

    agent.episode += 1
    if ep % CFG.target_update == 0:
        agent.sync_target()

    ep_ret = (train_env.final_value - CFG.initial_cash) / CFG.initial_cash * 100
    ep_returns.append(ep_ret)
    ep_losses.append(total_loss / max(steps, 1))
    ep_epsilons.append(agent.epsilon)

    if ep_ret > best_return:
        best_return = ep_ret
        agent.save('best_model.pt')

    if ep % 20 == 0 or ep == 1:
        print(f'Ep {ep:>4}/{CFG.episodes} | '
              f'Return: {ep_ret:+7.2f}% | '
              f'Best: {best_return:+7.2f}% | '
              f'epsilon: {agent.epsilon:.3f} | '
              f'Loss: {ep_losses[-1]:.5f} | '
              f'Trades: {train_env.trades}')

print('\nLoading best model…')
agent.load('best_model.pt')


## 9 · Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
ep = range(1, len(ep_returns) + 1)

ax = axes[0]
ax.plot(ep, ep_returns, alpha=0.3, color='steelblue', lw=1)
ax.plot(ep, pd.Series(ep_returns).rolling(20).mean(), color='steelblue', lw=2, label='20-ep MA')
ax.axhline(0, color='gray', ls='--', lw=0.8)
ax.set_title('Training Returns (%)'); ax.set_xlabel('Episode'); ax.legend()

ax = axes[1]
ax.plot(ep, ep_losses, alpha=0.4, color='crimson', lw=1)
ax.plot(ep, pd.Series(ep_losses).rolling(20).mean(), color='crimson', lw=2)
ax.set_title('Training Loss'); ax.set_xlabel('Episode')

ax = axes[2]
ax.plot(ep, ep_epsilons, color='darkorange', lw=2)
ax.set_title('Epsilon Decay'); ax.set_xlabel('Episode')
ax.set_ylim(0, 1)

plt.suptitle(f'Training Curves — {CFG.ticker}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 10 · Evaluation on Test Set

Runs the trained agent with ε = 0 (pure exploitation) on the held-out test period.

In [ ]:
agent.policy.eval()
test_env   = StockEnv(test_df, CFG)
state      = test_env.reset()
action_log = []
done       = False

while not done:
    with torch.no_grad():
        s      = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
        action = int(agent.policy(s).argmax(dim=1).item())
    state, _, done = test_env.step(action)
    action_log.append(action)

prices   = test_df['close'].values[CFG.window_size:]
bh_final = CFG.initial_cash * (prices[-1] / prices[0])
bh_ret   = (bh_final - CFG.initial_cash) / CFG.initial_cash * 100
rl_final = test_env.final_value
rl_ret   = (rl_final - CFG.initial_cash) / CFG.initial_cash * 100

portfolio_hist = np.array(test_env.history)
daily_ret = np.diff(portfolio_hist) / portfolio_hist[:-1]
sharpe    = (daily_ret.mean() / (daily_ret.std() + 1e-9)) * np.sqrt(252)
peak      = np.maximum.accumulate(portfolio_hist)
max_dd    = ((portfolio_hist - peak) / (peak + 1e-9)).min() * 100

print('=' * 55)
print('  EVALUATION RESULTS (Test Set)')
print('=' * 55)
print(f'  RL  Agent  : ${rl_final:>10,.2f}  ({rl_ret:+.2f}%)')
print(f'  Buy & Hold : ${bh_final:>10,.2f}  ({bh_ret:+.2f}%)')
print(f'  Sharpe     : {sharpe:.3f}')
print(f'  Max Drawdown: {max_dd:.2f}%')
print(f'  # Trades   : {test_env.trades}')
print('=' * 55)


## 11 · Evaluation Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
t = range(len(portfolio_hist))

ax = axes[0]
bh_curve = CFG.initial_cash * prices / prices[0]
ax.plot(t, portfolio_hist, color='seagreen', lw=2, label=f'RL Agent ({rl_ret:+.1f}%)')
ax.plot(t, bh_curve[:len(portfolio_hist)], color='royalblue', lw=2, ls='--',
        label=f'Buy & Hold ({bh_ret:+.1f}%)')
ax.axhline(CFG.initial_cash, color='gray', ls=':', lw=1)
ax.set_title('Portfolio Value (Test Set)'); ax.set_xlabel('Step')
ax.set_ylabel('Portfolio ($)'); ax.legend()

ax = axes[1]
labels = ['Hold', 'Buy 50%', 'Sell 50%', 'Buy All', 'Sell All']
counts = [action_log.count(i) for i in range(StockEnv.N_ACTIONS)]
colors = ['#aaaaaa', '#2ecc71', '#e74c3c', '#27ae60', '#c0392b']
bars   = ax.bar(labels, counts, color=colors, edgecolor='white')
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(cnt), ha='center', va='bottom', fontsize=9)
ax.set_title('Action Distribution'); ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=20)

ax = axes[2]
drawdown = (portfolio_hist - peak) / (peak + 1e-9) * 100
ax.fill_between(t, drawdown, 0, color='salmon', alpha=0.7)
ax.plot(t, drawdown, color='red', lw=1)
ax.set_title(f'Drawdown (Max: {max_dd:.1f}%)')
ax.set_xlabel('Step'); ax.set_ylabel('Drawdown (%)')

plt.suptitle(f'Evaluation — {CFG.ticker}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()


## 12 · Save / Load Model

In [ ]:
# The best checkpoint is already saved as best_model.pt during training.
# Re-run this cell any time to save the current policy explicitly.
agent.save('best_model.pt')

# To reload in a new session:
# new_agent = DQNAgent(StockEnv(train_df, CFG).obs_size, CFG)
# new_agent.load('best_model.pt')
